# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**My method is Logistic Regression**


My question isn't that "is this page good or bad?" (a fixed label), it is "which pages should a reviewer check first?". That's a ranking question, not a classification question. Because of this, I'm using a classifier's predicted probability, not its hard yes/no prediction, the probability gives me a natural ranking (higher probability = check first), the same way my ctr_gap score did in ML-07.


I'm starting with Logistic Regression specifically because it's the simplest and most readable option. I can see which features push a prediction up or down, so I want to understand what the model is doing before trying anything more complex like Random Forest. If Logistic Regression isn't good enough, that's a real finding too, it'll tell me about the pattern if it is more complicated than a straight line, and I'll have a real reason to try something stronger.


In [22]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)].copy()

tier_median_ctr = visible.groupby('position_tier')['ctr'].transform('median')
visible['ctr_gap'] = tier_median_ctr - visible['ctr']

# proxy label: is this page in the worst 25% of CTR gaps? (This is a real observed gap, not someone else's rule)
gap_threshold = visible['ctr_gap'].quantile(0.75)
visible['needs_review'] = (visible['ctr_gap'] >= gap_threshold).astype(int)

print(f"gap threshold (75th percentile): {gap_threshold:.3f}")
print(visible['needs_review'].value_counts())

gap threshold (75th percentile): 0.110
needs_review
0    8880
1    3143
Name: count, dtype: int64


In [23]:
print(visible.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'ctr_gap', 'needs_review']


In [24]:
feature_cols = [
    'search_volume', 'competition', 'cpc', 'content_type', 'main_intent',
    'word_count', 'char_count', 'provider_used', 'model_used',
    'impressions_90d', 'days_with_impressions', 'content_age_days',
    'days_since_last_update', 'avg_position', 'position_tier',
    'impression_tier', 'freshness_tier', 'age_tier', 'word_count_tier', 'char_count_tier'
]

X = visible[feature_cols].copy()
y = visible['needs_review']

print(X.shape, y.shape)
print(X.dtypes)

(12023, 20) (12023,)
search_volume             float64
competition               float64
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
days_with_impressions       int64
content_age_days            int64
days_since_last_update      int64
avg_position              float64
position_tier              object
impression_tier            object
freshness_tier             object
age_tier                   object
word_count_tier            object
char_count_tier            object
dtype: object


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**My split:** grouped by **client_id**, not a plain random split.

If I split the rows randomly, the same client may end up with some pages in training and some pages in testing. The model will then partly learn that client's specific patterns instead of learning something general, which will make my test score look better than what the model actually deserves. Splitting by client_id means every page from a given client stays entirely on one side, so the test set is genuinely unseen data.

In [25]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=visible['client_id']))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"train: {len(X_train):,} rows, {y_train.mean():.2%} needs_review")
print(f"test: {len(X_test):,} rows, {y_test.mean():.2%} needs_review")
print(f"train clients: {visible.iloc[train_idx]['client_id'].nunique()}, test clients: {visible.iloc[test_idx]['client_id'].nunique()}")
print(f"overlap in clients between train/test: {set(visible.iloc[train_idx]['client_id']) & set(visible.iloc[test_idx]['client_id'])}")

train: 11,202 rows, 26.33% needs_review
test: 821 rows, 23.63% needs_review
train clients: 22, test clients: 6
overlap in clients between train/test: set()


**Result**: 22 clients has 11,202 rows in train, 6 clients has 821 rows in test, with zero client overlap between them (confirmed by the empty set printed above). The test set is small since it is split by client count, not row count. Some clients simply have far more pages than others. I'll keep this in mind when reading my test results that a small test set means more room for the numbers to swing around by chance.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Training and comparing:** I'll train Logistic Regression on my train set and get predicted probabilities on the test set. I'll rank test pages by that probability, and compute precision@K, same metric I named back in ML-03. I'll compute the exact same precision@K for my ML-07 baseline's ranking (sorted by ctr_gap) on the same test rows, so both numbers come from an identical slice of data.

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_cols = ['content_type', 'main_intent', 'provider_used', 'model_used',
                     'position_tier', 'impression_tier', 'freshness_tier',
                     'age_tier', 'word_count_tier', 'char_count_tier']
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

X_train_filled = X_train.copy()
X_test_filled = X_test.copy()

X_train_filled[numeric_cols] = X_train_filled[numeric_cols].fillna(0)
X_test_filled[numeric_cols] = X_test_filled[numeric_cols].fillna(0)
X_train_filled[categorical_cols] = X_train_filled[categorical_cols].fillna('unknown').astype(str)
X_test_filled[categorical_cols] = X_test_filled[categorical_cols].fillna('unknown').astype(str)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols),
])

model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train_filled, y_train)
print("model trained, no warnings below this line means it converged cleanly")

model trained, no warnings below this line means it converged cleanly


In [27]:
# getting the model's predicted probability for each test page
test_probs = model.predict_proba(X_test_filled)[:, 1]

results = visible.iloc[test_idx].copy()
results['model_score'] = test_probs

def precision_at_k(df, score_col, k):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k['needs_review'].mean()

K = 50
baseline_p_at_k = precision_at_k(results, 'ctr_gap', K)
model_p_at_k = precision_at_k(results, 'model_score', K)
base_rate = results['needs_review'].mean()

comparison = pd.DataFrame({
    'method': ['base rate (random)', 'baseline (ctr_gap rule)', 'model (logistic regression)'],
    f'precision@{K}': [base_rate, baseline_p_at_k, model_p_at_k]
})
print(f"test set size: {len(results)}")
comparison

test set size: 821


,method,precision@50
0,base rate (random),0.236297
1,baseline (ctr_gap rule),1.000000
2,model (logistic regression),0.400000


In [28]:
naive_p_at_k = precision_at_k(results, 'avg_position', K)  # note: worse (higher) position = flagged first, so we sort ascending isn't right — need to flip sign
results['naive_score'] = -results['avg_position']  # flip so "worse position" ranks higher
naive_p_at_k = precision_at_k(results, 'naive_score', K)

comparison = pd.DataFrame({
    'method': ['base rate (random)', 'naive (worst position first)', 'baseline (ctr_gap rule — NOTE: circular, see below)', 'model (logistic regression)'],
    f'precision@{K}': [base_rate, naive_p_at_k, baseline_p_at_k, model_p_at_k]
})
comparison

,method,precision@50
0,base rate (random),0.236297
1,naive (worst position first),0.340000
2,"baseline (ctr_gap rule — NOTE: circular, see b...",1.000000
3,model (logistic regression),0.400000


**Important catch: my baseline comparison was circular.**

My **baseline** row scored a perfect **1.000 precision@50**. This should have felt too good to be true, and it was. My 'needs_review' label is defined from ctr_gap (top 25% of gaps = 1), and my baseline ranks pages by that exact same ctr_gap number. So the baseline wasn't predicting anything, it was just reading back the number that already defines the label. This is not a real test, it is circular.

For an honest comparison, I added a naive baseline that doesn't use ctr_gap either, sorting by worst avg_position alone. This gives every method a fair, non-circular test.

**Honest results at precision@50:**
- Base rate (random): 0.24
- Naive (worst position first): 0.34
- Model (Logistic Regression): 0.40

**Verdict:** The model genuinely beats both fair comparisons. It found real signal beyond just 'is this page ranked badly', using features like search_volume content_type, and freshness without ever seeing ctr or ctr_gap directly. This is a real, defensible win, even though it's a smaller margin than the fake 1.000 number suggested.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What I'm checking here:** A precision number alone can't tell if the model is doing something sensible. I want to see which features it leans on the most. Do they make real-world sense, or look suspicious? And look at specific pages it got wrong to understand why they're hard cases.

In [29]:
feature_names = model.named_steps['prep'].get_feature_names_out()
coefs = model.named_steps['clf'].coef_[0]

importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
importance['abs_coef'] = importance['coefficient'].abs()
top_features = importance.sort_values('abs_coef', ascending=False).head(10)
print(top_features[['feature', 'coefficient']])

                                 feature  coefficient
26            cat__freshness_tier_91-180     1.490781
23              cat__freshness_tier_0-30    -1.488119
0   cat__content_type_comparison article     1.363261
49           num__days_since_last_update    -1.303409
1       cat__content_type_feedly article    -1.081417
28                   cat__age_tier_31-90    -1.024617
19              cat__position_tier_top_3     0.914513
29                    cat__age_tier_365+     0.818908
17           cat__position_tier_page_3_5    -0.793943
18           cat__position_tier_striking    -0.737744


In [30]:
results['predicted_label'] = (results['model_score'] >= 0.5).astype(int)
wrong = results[results['predicted_label'] != results['needs_review']].copy()
wrong['confidence'] = wrong['model_score'].apply(lambda p: max(p, 1 - p))
worst_wrong = wrong.sort_values('confidence', ascending=False).head(3)

cols = ['content_id', 'position_tier', 'avg_position', 'ctr_gap', 'needs_review', 'model_score', 'freshness_tier', 'content_type']
worst_wrong[cols]

,content_id,position_tier,avg_position,ctr_gap,needs_review,model_score,freshness_tier,content_type
19912,content_7f96b461f858,striking,17.3,0.15,1,0.060469,0-30,keyword article
26898,content_d71d13f656da,page_1,7.8,0.12,1,0.073357,0-30,keyword article
2162,content_59d4182f3836,page_1,9.9,0.24,1,0.093920,0-30,keyword article


**Row 1** (`content_7f96b461f858`):

striking,
position 17.3,
ctr_gap 0.15

The model predicted only 0.06 probability of needing review. This is very confident.

Why it's hard: Because the page was updated in the last 30 days, and the model has learned that "recently updated" strongly predicts "no problem". freshness_tier_ (0-30) had a coefficient of -1.49. But this page still has a real gap. A recent update doesn't fix everything, and at position 17 (deep into striking distance), the page may need more than a refresh to actually climb.


**Row 2** (`content_d71d13f656da`):

page_1, position 7.8, ctr_gap 0.12

The model predicted only 0.07 probability of needing review. Again very confident.

Why it's hard: same pattern as row 1. freshness_tier is 0-30, so the model leaned hard on "recently updated = safe" and missed the real gap. This page ranks well (position 7.8, solidly on page 1), which likely reinforced the model's confidence even further. A well-ranked, recently-updated page looks like a success story on paper, but the CTR gap says otherwise.


**Row 3** (`content_59d4182f3836`):

page_1, position 9.9, ctr_gap 0.24 

the model predicted only 0.09 probability. The most confident wrong prediction of the three. And also the biggest real gap (0.24, the largest of the three misses).

Why it's hard: this is the clearest failure case. A genuinely large, real underperformance problem, but the model's trust in freshness_tier (0-30) overrode everything else and it missed the page almost entirely.


**What this tells overall:** All three wrong predictions share the same root cause that is that the model **over-relies** on freshness as a proxy for "this page is fine." A recent update doesn't guarantee good CTR, it just means someone touched the page recently. If I were to improve this model, I'd want a feature that actually captures what changed in the update (title change? content length change?), not just when it happened. Freshness alone is too blunt as a signal on its own.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.